# Evaluation: query Types 1–6

This notebook is the **main evaluation step** in the rerun pipeline.

It loads per-type qrels and queries, evaluates **RRF** (primary), **BM25** (mandatory baseline), and optional **dense** runs, then exports CSV summaries under `results/evaluation/`.

Metric policy is defined in `src/evaluation/utils.py` (`TYPE_METRICS`, `OVERALL_METRICS`).

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:

    def display(obj):
        print(obj)


# Project root (works if cwd is repo root or notebooks/)
ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "qrels").exists() and (ROOT.parent / "data" / "qrels").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.utils import (
    OVERALL_METRICS,
    TYPE_METRICS,
    build_bm25_rrf_comparison_table,
    default_data_paths,
    evaluate_all_types,
    evaluate_overall_types_1_6,
    evaluate_per_query_all_types,
    load_run_csv,
    merge_all_types_qrels,
    merge_all_types_queries,
    round_metrics_df,
    validate_run_coverage,
)

print("PROJECT_ROOT =", ROOT)
print("TYPE_METRICS keys:", sorted(TYPE_METRICS.keys()))

PROJECT_ROOT = /Users/alireza/Library/CloudStorage/GoogleDrive-ali.esterabi@gmail.com/My Drive/QMUL_temr_2/IR_project
TYPE_METRICS keys: [1, 2, 3, 4, 5, 6]


## Configuration

Set paths to your ranked-list CSVs. Expected columns: `query_id`, `doc_id`, `rank` (and for RRF: `rrf_score`; for BM25 often `score` or `bm25_score`).

In [2]:
# --- Required: canonical fused run and BM25 baseline ---
RRF_RUN_PATH = ROOT / "results" / "runs" / "rrf_bge_m3.csv"
BM25_RUN_PATH = ROOT / "results" / "runs" / "bm25.csv"

# Optional: dense retrieval run (same CSV schema)
DENSE_RUN_PATH = ROOT / "results" / "runs" / "dense_bge_m3.csv"  # set to None to skip

QRELS_PATHS, QUERY_PATHS = default_data_paths(ROOT)
# Only Types 1–6
QRELS_PATHS = {t: QRELS_PATHS[t] for t in range(1, 7)}
QUERY_PATHS = {t: QUERY_PATHS[t] for t in range(1, 7)}

OUT_DIR = ROOT / "results" / "evaluation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

for label, p in [("RRF", RRF_RUN_PATH), ("BM25", BM25_RUN_PATH)]:
    print(f"{label}: {p}  exists={p.exists()}")
if DENSE_RUN_PATH:
    print(f"Dense: {DENSE_RUN_PATH}  exists={Path(DENSE_RUN_PATH).exists()}")

RRF: /Users/alireza/Library/CloudStorage/GoogleDrive-ali.esterabi@gmail.com/My Drive/QMUL_temr_2/IR_project/results/runs/rrf_bge_m3.csv  exists=True
BM25: /Users/alireza/Library/CloudStorage/GoogleDrive-ali.esterabi@gmail.com/My Drive/QMUL_temr_2/IR_project/results/runs/bm25.csv  exists=True
Dense: /Users/alireza/Library/CloudStorage/GoogleDrive-ali.esterabi@gmail.com/My Drive/QMUL_temr_2/IR_project/results/runs/dense_bge_m3.csv  exists=True


## Load qrels and queries (Types 1–6)

In [3]:
qrels = merge_all_types_qrels(QRELS_PATHS)
queries_df = merge_all_types_queries(QUERY_PATHS)

print("Total qrels queries:", len(qrels))
print("Total query rows:", len(queries_df))
print(queries_df["query_type"].value_counts().sort_index())

Total qrels queries: 228
Total query rows: 232
query_type
1    50
2    50
3    18
4    14
5    50
6    50
Name: count, dtype: int64


## Load runs

In [4]:
run_rrf = load_run_csv(RRF_RUN_PATH, score_col="rrf_score")
run_bm25 = load_run_csv(BM25_RUN_PATH)

run_dense = None
dense_by_type = None
if DENSE_RUN_PATH and Path(DENSE_RUN_PATH).exists():
    run_dense = load_run_csv(DENSE_RUN_PATH)

print("RRF queries in run:", len(run_rrf))
print("BM25 queries in run:", len(run_bm25))

RRF queries in run: 232
BM25 queries in run: 174


## Validation: coverage vs Types 1–6 query IDs (in qrels)

In [5]:
mask = queries_df["query_type"].between(1, 6)
eval_qids = [
    q for q in queries_df.loc[mask, "query_id"].astype(str).tolist() if q in qrels
]

cov_rrf = validate_run_coverage(run_rrf, eval_qids, min_depth=10)
cov_bm25 = validate_run_coverage(run_bm25, eval_qids, min_depth=10)
print("RRF missing queries:", cov_rrf["missing"].sum())
print("BM25 missing queries:", cov_bm25["missing"].sum())
display(cov_rrf[cov_rrf["missing"] | cov_rrf["shallow"]].head(20))

RRF missing queries: 0
BM25 missing queries: 58


,query_id,n_docs,missing,shallow


## Primary system: RRF — per-type, overall, per-query

In [6]:
rrf_by_type = evaluate_all_types(qrels, run_rrf, queries_df)
rrf_overall = evaluate_overall_types_1_6(qrels, run_rrf, queries_df)
rrf_per_query = evaluate_per_query_all_types(qrels, run_rrf, queries_df)

display(round_metrics_df(rrf_by_type))
print("RRF overall (Types 1–6):", rrf_overall)

,query_type,n_queries,map,mrr,ndcg@10,precision@1,recall@10,recall@20
0,1,50,0.0018,0.0018,NaN,0.00,NaN,NaN
1,2,50,0.9822,0.9822,NaN,0.98,NaN,NaN
2,3,14,0.0220,NaN,0.0394,NaN,0.0574,NaN
3,4,14,0.0218,NaN,0.0328,NaN,0.0357,NaN
4,5,50,0.0000,NaN,NaN,NaN,0.0000,0.0000
5,6,50,0.0182,NaN,NaN,NaN,0.0070,0.0129


RRF overall (Types 1–6): {'recall@10': np.float64(0.22654215736095082), 'map': np.float64(0.22247549767316363)}


## Baseline: BM25 — per-type, overall, per-query

In [7]:
bm25_by_type = evaluate_all_types(qrels, run_bm25, queries_df)
bm25_overall = evaluate_overall_types_1_6(qrels, run_bm25, queries_df)
bm25_per_query = evaluate_per_query_all_types(qrels, run_bm25, queries_df)

display(round_metrics_df(bm25_by_type))
print("BM25 overall (Types 1–6):", bm25_overall)

,query_type,n_queries,map,mrr,ndcg@10,precision@1,recall@10,recall@20
0,1,50,0.0000,0.0,NaN,0.0,NaN,NaN
1,2,50,1.0000,1.0,NaN,1.0,NaN,NaN
2,3,14,0.0718,NaN,0.1355,NaN,0.0758,NaN
3,4,14,0.0315,NaN,0.0550,NaN,0.0568,NaN
4,5,50,0.0000,NaN,NaN,NaN,0.0000,0.0000
5,6,50,0.0058,NaN,NaN,NaN,0.0016,0.0043


BM25 overall (Types 1–6): {'recall@10': np.float64(0.22779081230641457), 'map': np.float64(0.22691739668362054)}


## Report-critical: BM25 vs RRF (deltas per metric)

In [8]:
compare_by_type = build_bm25_rrf_comparison_table(bm25_by_type, rrf_by_type)
display(round_metrics_df(compare_by_type))

overall_compare = pd.DataFrame(
    [{"system": "BM25", **bm25_overall}, {"system": "RRF", **rrf_overall}]
)
delta = {"system": "RRF_minus_BM25"}
for m in OVERALL_METRICS:
    if m in bm25_overall and m in rrf_overall:
        delta[m] = rrf_overall[m] - bm25_overall[m]
overall_compare = pd.concat(
    [overall_compare, pd.DataFrame([delta])], ignore_index=True
)
display(round_metrics_df(overall_compare))

,query_type,n_queries,map_RRF_minus_BM25,map_bm25,map_rrf,mrr_RRF_minus_BM25,mrr_bm25,mrr_rrf,ndcg@10_RRF_minus_BM25,ndcg@10_bm25,ndcg@10_rrf,precision@1_RRF_minus_BM25,precision@1_bm25,precision@1_rrf,recall@10_RRF_minus_BM25,recall@10_bm25,recall@10_rrf,recall@20_RRF_minus_BM25,recall@20_bm25,recall@20_rrf
0,1,50,0.0018,0.0000,0.0018,0.0018,0.0,0.0018,NaN,NaN,NaN,0.00,0.0,0.00,NaN,NaN,NaN,NaN,NaN,NaN
1,2,50,-0.0178,1.0000,0.9822,-0.0178,1.0,0.9822,NaN,NaN,NaN,-0.02,1.0,0.98,NaN,NaN,NaN,NaN,NaN,NaN
2,3,14,-0.0498,0.0718,0.0220,NaN,NaN,NaN,-0.0961,0.1355,0.0394,NaN,NaN,NaN,-0.0184,0.0758,0.0574,NaN,NaN,NaN
3,4,14,-0.0097,0.0315,0.0218,NaN,NaN,NaN,-0.0222,0.0550,0.0328,NaN,NaN,NaN,-0.0211,0.0568,0.0357,NaN,NaN,NaN
4,5,50,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
5,6,50,0.0123,0.0058,0.0182,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0054,0.0016,0.0070,0.0086,0.0043,0.0129


,system,recall@10,map
0,BM25,0.2278,0.2269
1,RRF,0.2265,0.2225
2,RRF_minus_BM25,-0.0012,-0.0044


## Optional: dense run (same evaluation hooks)

In [9]:
if run_dense:
    dense_by_type = evaluate_all_types(qrels, run_dense, queries_df)
    display(round_metrics_df(dense_by_type))
else:
    print("No dense run loaded — skipped.")
    dense_by_type = None

,query_type,n_queries,map,mrr,ndcg@10,precision@1,recall@10,recall@20
0,1,50,0.0018,0.0018,NaN,0.00,NaN,NaN
1,2,50,0.8636,0.8636,NaN,0.82,NaN,NaN
2,3,14,0.0004,NaN,0.0000,NaN,0.0000,NaN
3,4,14,0.0089,NaN,0.0176,NaN,0.0179,NaN
4,5,50,0.0000,NaN,NaN,NaN,0.0000,0.0000
5,6,50,0.0211,NaN,NaN,NaN,0.0096,0.0152


## Export CSVs

Files go to `results/evaluation/` so the notebook writes the reusable evaluation tables without generating a separate Markdown report.


In [10]:
round_metrics_df(rrf_by_type).to_csv(OUT_DIR / "types_1_6_rrf_by_type.csv", index=False)
pd.DataFrame([rrf_overall]).to_csv(OUT_DIR / "types_1_6_rrf_overall.csv", index=False)
rrf_per_query.to_csv(OUT_DIR / "types_1_6_rrf_per_query.csv", index=False)

round_metrics_df(bm25_by_type).to_csv(OUT_DIR / "types_1_6_bm25_by_type.csv", index=False)
pd.DataFrame([bm25_overall]).to_csv(OUT_DIR / "types_1_6_bm25_overall.csv", index=False)
bm25_per_query.to_csv(OUT_DIR / "types_1_6_bm25_per_query.csv", index=False)

round_metrics_df(compare_by_type).to_csv(OUT_DIR / "types_1_6_bm25_vs_rrf_by_type.csv", index=False)

# Overall BM25 vs RRF row
ov_comp = pd.DataFrame(
    [
        {"system": "BM25", **bm25_overall},
        {"system": "RRF", **rrf_overall},
    ]
)
if len(ov_comp) == 2:
    delta = {"system": "RRF_minus_BM25"}
    for m in OVERALL_METRICS:
        delta[m] = ov_comp.loc[1, m] - ov_comp.loc[0, m]
    ov_comp = pd.concat([ov_comp, pd.DataFrame([delta])], ignore_index=True)
round_metrics_df(ov_comp).to_csv(OUT_DIR / "types_1_6_bm25_vs_rrf_overall.csv", index=False)

if dense_by_type is not None:
    round_metrics_df(dense_by_type).to_csv(
        OUT_DIR / "types_1_6_dense_by_type.csv", index=False
    )

print("Wrote CSVs to", OUT_DIR)

Wrote CSVs to /Users/alireza/Library/CloudStorage/GoogleDrive-ali.esterabi@gmail.com/My Drive/QMUL_temr_2/IR_project/results/evaluation


## Notes

- **Types 3–4** use graded qrels; `ndcg@10` is the primary ranked metric for those types.
- **Types 5–6** emphasize recall at shallow depth.
- Unjudged documents are not in qrels; standard pooling assumption applies for retrieved unjudged docs.
- If a run file is missing, fix `RRF_RUN_PATH` / `BM25_RUN_PATH` above (or generate runs from your retrieval pipeline).